# Analyse multi-nœuds — modèle SEAM

Notebook de post-traitement pour l'ensemble des campagnes `ep` (tous nœuds, toutes architectures).
Les données source sont dans `resultats/parallel_sim/` ; les scripts de figure sont dans `postprocess/`.

## Modèle SEAM

$$E(n_c) = e_{\text{core}} + \frac{e_{\text{uncore}}}{n_c} \quad \Rightarrow \quad S_e = \frac{e_{\text{uncore}}}{e_{\text{core}}}$$

Deux estimateurs :
- **$S_{e,\text{OLS}}$** — régression OLS sur les moyennes par $n_c$ ($n_c \geq 2$)
- **$\hat{S}_e(n_c)$** — estimateur par deux points : $E(1)$ extrapolé OLS et $E_{\text{med}}(n_c)$ mesuré

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from scipy.stats import shapiro

BASE = Path('..') / 'resultats' / 'parallel_sim'
OUT  = Path('figures')
OUT.mkdir(exist_ok=True)

# ── Architecture colors (palette identique à docs/postprocess/se_per_node.py) ─
ARCH_COLOR = {
    'SNB': '#2a78d6', 'IVB': '#4a3aa7', 'HSW': '#eb6834',
    'BDW': '#1baf7a', 'CLX': '#eda100', 'SKL': '#e87ba4',
    'ICX': '#e34948', 'EMR': '#6e4e37', 'ZEN1': '#7b2d8b', 'ZEN3': '#b5651d',
}
ARCH_ORDER = ['IVB', 'SNB', 'HSW', 'BDW', 'CLX', 'SKL', 'ICX', 'EMR', 'ZEN1', 'ZEN3']

# ── Nœuds disponibles ─────────────────────────────────────────────────────────
NODES = [
    ('IVB', 'gof17',        'parallel_sim_ivb_gof17.csv'),
    ('SNB', 'econome-6',    'parallel_sim_snb_econome-6.csv'),
    ('SNB', 'orion-4',      'parallel_sim_snb_orion-4.csv'),
    ('SNB', 'taurus-3',     'parallel_sim_snb_taurus-3.csv'),
    ('SNB', 'taurus-4',     'parallel_sim_snb_taurus-4.csv'),
    ('SNB', 'hercule-1',    'parallel_sim_snb_hercule-1.csv'),
    ('SNB', 'taurus-6',     'parallel_sim_snb_taurus-6.csv'),
    ('SNB', 'taurus-7',     'parallel_sim_snb_taurus-7.csv'),
    ('HSW', 'parasilo-2',   'parallel_sim_hsw_parasilo-2.csv'),
    ('BDW', 'nova-1',       'parallel_sim_bdw_nova-1.csv'),
    ('BDW', 'ecotype-2',    'parallel_sim_bdw_ecotype-2.csv'),
    ('BDW', 'clervaux-8',   'parallel_sim_bdw_clervaux-8.csv'),
    ('CLX', 'gros-1',       'parallel_sim_clx_gros-1_fixed.csv'),
    ('SKL', 'dahu-1',       'parallel_sim_skl_dahu-1.csv'),
    ('SKL', 'chifflot-1',   'parallel_sim_skl_chifflot-1.csv'),
    ('ICX', 'paradoxe-2',   'parallel_sim_icx_paradoxe-2.csv'),
    ('ICX', 'chirop-1',     'parallel_sim_icx_chirop-1.csv'),
    ('ICX', 'chirop-4',     'parallel_sim_icx_chirop-4.csv'),
    ('ICX', 'chirop-5',     'parallel_sim_icx_chirop-5.csv'),
    ('ICX', 'spirou-3',     'parallel_sim_icx_spirou-3.csv'),
    ('ICX', 'montcalm-1',   'parallel_sim_icx_montcalm-1.csv'),
    ('ICX', 'fleckenstein-6','parallel_sim_icx_fleckenstein-6.csv'),
    ('EMR', 'larochette-1', 'parallel_sim_emr_larochette-1.csv'),
    ('EMR', 'larochette-4', 'parallel_sim_emr_larochette-4.csv'),
    ('ZEN1','chiclet-1',    'parallel_sim_zen1_chiclet-1.csv'),
    ('ZEN3','chuc-1',       'parallel_sim_zen3_chuc-1.csv'),
]

## 1. Fonctions SEAM

In [ ]:
def fit_seam_csv(filepath, seq_frac=0.0):
    """
    OLS sur E(n_c) = ecore_c + euncore_c/n_c  (n_c >= 2, seq_frac=0).
    Lit le CSV directement (colonnes N_core / Energy / Seq_frac).
    Retourne dict(Se, ecore_c, euncore_c, n_max, rmse, n_points).
    """
    rows = {}
    with open(filepath) as f:
        for r in csv.DictReader(f):
            if float(r.get('Seq_frac', r.get('seq_fraction', 0))) != seq_frac:
                continue
            nc_key = 'N_core' if 'N_core' in r else 'N_cores'
            n = int(float(r[nc_key]))
            if n < 2:
                continue
            e_key = 'Energy' if 'Energy' in r else 'Energy_J'
            rows.setdefault(n, []).append(float(r[e_key]))
    ns  = np.array(sorted(rows), dtype=float)
    E_m = np.array([np.mean(rows[n]) for n in ns])
    X   = np.column_stack([np.ones(len(ns)), 1.0 / ns])
    ecore_c, euncore_c = np.linalg.lstsq(X, E_m, rcond=None)[0]
    rmse = float(np.sqrt(np.mean((ecore_c + euncore_c / ns - E_m) ** 2)))
    return dict(Se=euncore_c / ecore_c,
                ecore_c=ecore_c, euncore_c=euncore_c,
                n_max=int(ns.max()), rmse=rmse, n_points=len(ns))


def se_inv_csv(filepath, fit, seq_frac=0.0):
    """
    Estimateur par deux points par palier : Se_inv(n_c) = n_c*(G-1)/(n_c-G)
    avec G = E_ref / E_med(n_c) et E_ref = ecore_c + euncore_c.
    """
    E_ref = fit['ecore_c'] + fit['euncore_c']
    rows = {}
    with open(filepath) as f:
        for r in csv.DictReader(f):
            if float(r.get('Seq_frac', r.get('seq_fraction', 0))) != seq_frac:
                continue
            nc_key = 'N_core' if 'N_core' in r else 'N_cores'
            n = int(float(r[nc_key]))
            if n < 2:
                continue
            e_key = 'Energy' if 'Energy' in r else 'Energy_J'
            rows.setdefault(n, []).append(float(r[e_key]))
    out = []
    for nc, vals in rows.items():
        E_med = float(np.median(vals))
        G     = E_ref / E_med
        denom = nc - G
        if abs(denom) < 1e-6 or G <= 1:
            continue
        out.append({'N_core': nc, 'Se_inv': nc * (G - 1) / denom})
    return pd.DataFrame(out)

## 2. Calcul Se — tous nœuds

In [ ]:
results = []
for arch, label, fname in NODES:
    p = BASE / fname
    if not p.exists():
        print(f'[warn] manquant : {fname}')
        continue
    try:
        f = fit_seam_csv(p)
        results.append({'arch': arch, 'node': label, **f})
    except Exception as e:
        print(f'[warn] {label}: {e}')

df_se = pd.DataFrame(results)
print(f"{len(df_se)} nœuds traités\n")
print(df_se[['arch','node','Se','n_max','rmse','n_points']].to_string(index=False))

## 3. Se par nœud (figure principale)

In [ ]:
arch_groups = {a: df_se[df_se['arch'] == a].reset_index(drop=True)
               for a in ARCH_ORDER if a in df_se['arch'].values}

y_pos, group_spans = {}, {}
y = 0
for arch, grp in arch_groups.items():
    y_start = y - 0.5
    for _, row in grp.iterrows():
        y_pos[row['node']] = y; y += 1
    group_spans[arch] = (y_start, y - 0.5)
    y += 0.4

plt.rcParams.update({'font.size': 8.5, 'axes.linewidth': 0.6,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.spines.left': False, 'grid.linewidth': 0.35,
                     'grid.color': '#e1e0d9', 'legend.frameon': False})

fig, ax = plt.subplots(figsize=(10, max(6, len(df_se) * 0.45)))

for i, (arch, (ys, ye)) in enumerate(group_spans.items()):
    if i % 2 == 0:
        ax.axhspan(ys, ye, color='#f0efed', zorder=0, lw=0)

ax.grid(axis='x', zorder=1)
ax.axvline(0, color='#c3c2b7', lw=0.6, zorder=2)

for arch, grp in arch_groups.items():
    c = ARCH_COLOR.get(arch, '#888')
    for _, row in grp.iterrows():
        yi = y_pos[row['node']]
        ax.scatter(row['Se'], yi, s=55, color=c, zorder=5,
                   edgecolors='white', linewidths=0.8)
        ax.text(row['Se'] + 0.3, yi, f"({row['n_max']}c)",
                va='center', fontsize=6.5, color='#898781')

yticks  = [y_pos[r['node']] for _, r in df_se.iterrows()]
ylabels = [r['node'] for _, r in df_se.iterrows()]
ax.set_yticks(yticks); ax.set_yticklabels(ylabels, fontsize=7.5, color='#52514e')
ax.set_ylim(-0.7, y - 0.3)
ax.set_xlabel(r'Paramètre $S_e$ (régression OLS, $n \geq 2$)', labelpad=7)
ax.set_title(r'$S_e$ par nœud — modèle SEAM', fontsize=10, fontweight='semibold', pad=9)

patches = [mpatches.Patch(color=ARCH_COLOR.get(a,'#888'), label=a)
           for a in ARCH_ORDER if a in arch_groups]
ax.legend(handles=patches, loc='lower right', ncol=2, fontsize=7,
          handlelength=1, handleheight=0.8)

plt.tight_layout()
fig.savefig(OUT / 'se_per_node.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"Sauvegardé : {OUT / 'se_per_node.png'}")

## 4. Cohérence $S_{e,\mathrm{OLS}}$ vs $\hat{S}_e(n_c)$ — tous nœuds

In [ ]:
records = []  # (arch, node, Se_OLS, nc, Se_inv)
for arch, label, fname in NODES:
    p = BASE / fname
    if not p.exists():
        continue
    row = df_se[df_se['node'] == label]
    if row.empty:
        continue
    fit = row.iloc[0].to_dict()
    inv = se_inv_csv(p, fit)
    for _, r in inv.iterrows():
        records.append({'arch': arch, 'node': label,
                        'Se_OLS': fit['Se'], 'nc': r['N_core'], 'Se_inv': r['Se_inv']})

df_inv = pd.DataFrame(records)
Se_OLS_all = df_inv['Se_OLS'].values
Se_inv_all = df_inv['Se_inv'].values
rmse_global = float(np.sqrt(np.mean((Se_inv_all - Se_OLS_all) ** 2)))
print(f"{len(df_inv)} points  |  RMSE global = {rmse_global:.3f}")

fig, ax = plt.subplots(figsize=(7, 7))
lim = max(Se_OLS_all.max(), Se_inv_all.max()) * 1.08
ax.plot([0, lim], [0, lim], '--', color='#bbb', lw=1.2, label='y = x')
ax.fill_between([0, lim], [0, lim * 0.9], [0, lim * 1.1],
                color='#f0f0f0', alpha=0.6)
for arch in ARCH_ORDER:
    sub = df_inv[df_inv['arch'] == arch]
    if sub.empty:
        continue
    c = ARCH_COLOR.get(arch, '#888')
    ax.scatter(sub['Se_OLS'], sub['Se_inv'], s=20, color=c, alpha=0.55,
               edgecolors='white', linewidths=0.4, label=arch, zorder=3)
ax.text(0.04, 0.96,
        f'RMSE = {rmse_global:.2f}\n({len(df_inv)} points, {len(df_se)} nœuds)',
        transform=ax.transAxes, fontsize=9, va='top', color='#444',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#ddd', alpha=0.9))
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel('$S_{e,\\mathrm{OLS}}$ (régression)')
ax.set_ylabel('$\\hat{S}_e(n_c)$ (estimateur par deux points)')
ax.set_title('Cohérence interne des estimateurs SEAM\n(tous nœuds)')
ax.legend(fontsize=8, ncol=2); ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(OUT / 'estimateurs.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"Sauvegardé : {OUT / 'estimateurs.png'}")

## 5. Distribution de $S_e$ par architecture

In [ ]:
archs_present = [a for a in ARCH_ORDER if a in df_se['arch'].values]

fig, ax = plt.subplots(figsize=(9, 4))
for arch in archs_present:
    vals = df_se[df_se['arch'] == arch]['Se'].values
    c = ARCH_COLOR.get(arch, '#888')
    ax.scatter([arch] * len(vals), vals, color=c, s=55,
               edgecolors='white', linewidths=0.8, zorder=3)
    ax.plot([arch, arch], [vals.min(), vals.max()],
            color=c, lw=1.5, alpha=0.4, zorder=2)

ax.set_xlabel('Architecture'); ax.set_ylabel('$S_e$ (OLS)')
ax.set_title('Distribution de $S_e$ par architecture')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(OUT / 'se_distribution.png', dpi=200, bbox_inches='tight')
plt.show()

print(f"\n{'arch':<6} {'n':>4} {'Se moy':>9} {'Se min':>9} {'Se max':>9}")
print('-' * 42)
for arch in archs_present:
    sub = df_se[df_se['arch'] == arch]['Se']
    print(f"{arch:<6} {len(sub):>4} {sub.mean():>9.2f} {sub.min():>9.2f} {sub.max():>9.2f}")

## 6. Validation du protocole — stabilité CV(n_stat) et quantification RAPL

In [ ]:
# Architectures représentatives (un nœud par archi)
REPR = {
    'SNB': ('econome-6',  'parallel_sim_snb_econome-6.csv',  8,  15.3e-6),
    'BDW': ('ecotype-2',  'parallel_sim_bdw_ecotype-2.csv',  10, 15.3e-6),
    'SKL': ('dahu-1',     'parallel_sim_skl_dahu-1.csv',     16, 15.3e-6),
    'CLX': ('gros-1',     'parallel_sim_clx_gros-1_fixed.csv', 18, 15.3e-6),
    'ICX': ('chirop-1',   'parallel_sim_icx_chirop-1.csv',   32, 3.82e-6),
}
N_STAT_MAX = 30

cv_data, e_nmax, e_n2 = {}, {}, {}

for arch, (node, fname, ncmax, esu) in REPR.items():
    p = BASE / fname
    if not p.exists():
        print(f'[warn] manquant : {fname}'); continue
    vals_nmax, vals_n2 = [], []
    with open(p) as f:
        for r in csv.DictReader(f):
            if float(r.get('Seq_frac', 0)) != 0.0: continue
            e_key = 'Energy' if 'Energy' in r else 'Energy_J'
            nc_key = 'N_core' if 'N_core' in r else 'N_cores'
            n = int(float(r[nc_key]))
            e = float(r[e_key])
            if n == ncmax: vals_nmax.append(e)
            if n == 2:     vals_n2.append(e)
    # CV(n_stat) — 30 premières mesures de la séquence entrelacée à n_c=max
    sample = np.array(vals_nmax[:N_STAT_MAX])
    if len(sample) >= 2:
        cv_data[arch] = [np.std(sample[:k]) / np.mean(sample[:k]) * 100
                         for k in range(2, len(sample) + 1)]
    e_nmax[arch] = (np.mean(vals_nmax), esu)
    e_n2[arch]   = (np.mean(vals_n2),   esu)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Validation du protocole de mesure', fontsize=11)

ax = axes[0]
for arch, cv in cv_data.items():
    c = ARCH_COLOR.get(arch, '#888')
    ax.plot(range(2, 2 + len(cv)), cv, color=c, lw=1.5, label=arch)
ax.axhline(3, color='k', ls=':', lw=0.8, label='seuil 3 %')
ax.set_xlabel('n_stat'); ax.set_ylabel('CV(E) %')
ax.set_title('Stabilité statistique — CV(n_stat)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[1]
x = np.arange(len(REPR))
archs_r = list(REPR.keys())
for i, arch in enumerate(archs_r):
    if arch not in e_nmax: continue
    c = ARCH_COLOR.get(arch, '#888')
    e_nmax_v, esu = e_nmax[arch]
    e_n2_v, _     = e_n2[arch]
    ax.bar(i - 0.2, e_n2_v   / esu, 0.35, color=c, alpha=0.9, label=f'{arch} n=2')
    ax.bar(i + 0.2, e_nmax_v / esu, 0.35, color=c, alpha=0.45)
ax.set_yscale('log')
ax.set_xticks(x); ax.set_xticklabels(archs_r)
ax.set_ylabel('E / ESU RAPL'); ax.set_title('Marge quantification RAPL (>50 000 partout)')
ax.axhline(50000, color='k', ls=':', lw=0.8, label='seuil min 50k')
ax.legend(fontsize=7); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig(OUT / 'validation_nstat.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"Sauvegardé : {OUT / 'validation_nstat.png'}")

## 7. Sensibilité de $S_e$ à la taille de charge (work_sweep)

Cellule active quand les fichiers `resultats/{node}_work_sweep_nw*.csv` existent
(produits par `launch_scripts/parasilo_work_sweep.sh`).

In [ ]:
import glob

work_sweep_files = sorted(glob.glob(str(Path('..') / 'resultats' / '*_work_sweep_nw*.csv')))
if not work_sweep_files:
    print("Aucun fichier work_sweep trouvé — lancez d'abord launch_scripts/parasilo_work_sweep.sh")
else:
    ws_results = []
    for fpath in work_sweep_files:
        fname = Path(fpath).stem
        label = fname.split('_work_sweep_')[-1]  # nw500K, nw1M, …
        try:
            f = fit_seam_csv(fpath)
            ws_results.append({'nw_label': label, 'Se': f['Se'], 'rmse': f['rmse']})
        except Exception as e:
            print(f'[warn] {fname}: {e}')

    df_ws = pd.DataFrame(ws_results)
    print(df_ws.to_string(index=False))

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(range(len(df_ws)), df_ws['Se'], 'o-', color=ARCH_COLOR.get('HSW','C1'), lw=2)
    ax.set_xticks(range(len(df_ws)))
    ax.set_xticklabels(df_ws['nw_label'], rotation=30)
    ax.set_ylabel('$S_e$ (OLS)'); ax.set_xlabel('Taille de charge $n_{\\mathrm{work}}$')
    ax.set_title('Sensibilité de $S_e$ à $n_{\\mathrm{work}}$ — parasilo (HSW)')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(OUT / 'work_sweep.png', dpi=200, bbox_inches='tight')
    plt.show()

## 8. Export se_summary.csv

In [ ]:
out_csv = OUT / 'se_summary.csv'
df_se[['arch','node','Se','ecore_c','euncore_c','n_max','rmse','n_points']].to_csv(out_csv, index=False)
print(f"Exporté : {out_csv}")
display(df_se[['arch','node','Se','n_max','rmse']].sort_values(['arch','Se']))